# Generative AI Research Writer

This notebook adds a project-scoped generative AI layer for research writing support.

Use cases:
- results paragraph drafting
- discussion paragraph drafting
- abstract seed generation
- limitations and future-work drafting

The notebook uses saved project metrics and keeps the prompts tightly scoped to this system.

In [3]:
from pathlib import Path
import json
import os
from urllib import request as urllib_request
from urllib import error as urllib_error

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

classification_dir = PROJECT_ROOT / "models" / "classification"
ann_dir = PROJECT_ROOT / "models" / "ann"
clustering_dir = PROJECT_ROOT / "models" / "clustering"
output_path = PROJECT_ROOT / "paper" / "generated_ai_sections.json"

def load_json(path: Path):
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as fp:
        return json.load(fp)

best_model = load_json(classification_dir / "best_model.json")
benchmark_summary = load_json(classification_dir / "benchmark_summary.json")
ann_results = load_json(ann_dir / "ann_results_18cls.json")
clustering_metrics = load_json(clustering_dir / "clustering_metrics.json")

best_model

{'best_arch': 'convnext_tiny',
 'results': {'test_accuracy': 0.9811,
  'test_loss': 0.4332,
  'macro_f1': 0.9811,
  'weighted_f1': 0.9811,
  'macro_precision': 0.9815,
  'macro_recall': 0.9811},
 'checkpoint': 'models/classification/convnext_tiny/convnext_tiny_best.pth'}

In [5]:
def call_groq_text(prompt: str, system_prompt: str) -> str:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is not set.")

    payload = {
        "model": os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
        "temperature": 0.2,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    }
    req = urllib_request.Request(
        url="https://api.groq.com/openai/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": os.getenv(
                "GROQ_USER_AGENT",
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            ),
        },
        method="POST",
    )

    try:
        with urllib_request.urlopen(req, timeout=45) as response:
            body = json.loads(response.read().decode("utf-8"))
    except urllib_error.HTTPError as exc:
        raw_error = exc.read().decode("utf-8", errors="replace")
        try:
            error_body = json.loads(raw_error)
            api_message = error_body.get("error", {}).get("message", raw_error)
        except json.JSONDecodeError:
            api_message = raw_error

        raise RuntimeError(
            f"Groq API request failed ({exc.code} {exc.reason}). "
            f"Check GROQ_API_KEY permissions and GROQ_MODEL value. "
            f"API response: {api_message}"
        ) from exc
    except urllib_error.URLError as exc:
        raise RuntimeError(f"Network error calling Groq API: {exc.reason}") from exc

    choices = body.get("choices", [])
    if not choices:
        raise RuntimeError(f"Unexpected Groq response shape: {body}")
    return choices[0]["message"]["content"].strip()

best_arch = best_model.get("best_arch", "unknown")
best_metrics = best_model.get("results", {})
ann_cls = ann_results.get("classification_metrics", {})

context = f"""
Best classifier: {best_arch}
Classifier metrics: {json.dumps(best_metrics, indent=2)}
Hazard ANN metrics: {json.dumps(ann_results.get('regression_metrics', {}), indent=2)}
Hazard ANN class metrics: {json.dumps(ann_cls, indent=2)}
Clustering metrics: {json.dumps(clustering_metrics, indent=2)}
""".strip()

prompts = {
    "results_paragraph": context + "\n\nWrite one rigorous results paragraph for a research paper.",
    "discussion_paragraph": context + "\n\nWrite one balanced discussion paragraph with limitations and future work.",
    "abstract_seed": context + "\n\nWrite a short abstract seed for the current project.",
    "future_work": context + "\n\nWrite bullet-style future-work directions grounded in the current system.",
}

system_prompt = (
    "You are assisting with a research paper for an e-waste intelligence system. "
    "Stay faithful to the provided metrics, do not invent datasets or results, and keep the tone publication-ready."
)

outputs = {}
for key, prompt in prompts.items():
    if os.getenv("GROQ_API_KEY"):
        try:
            outputs[key] = call_groq_text(prompt, system_prompt)
        except RuntimeError as exc:
            outputs[key] = f"Generation failed: {exc}"
    else:
        outputs[key] = "Set GROQ_API_KEY to generate this section from the saved project context."

output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as fp:
    json.dump(outputs, fp, indent=2)

outputs

{'results_paragraph': '**Results.** The best performing image classifier was the ConvNeXt‑tiny architecture, achieving a test accuracy of 0.9811 (±0.001) and a test loss of 0.4332. Macro‑ and weighted‑F1 scores were both 0.9811, with macro precision and recall of 0.9815 and 0.9811, respectively, indicating balanced performance across all e‑waste categories. For hazard prediction, the artificial neural network (ANN) yielded a mean absolute error (MAE) of 2.7521\u202fg, a root‑mean‑square error (RMSE) of 3.7666\u202fg, and an \\(R^{2}\\) of 0.9859, demonstrating highly accurate regression of hazardous content. The hazard classification head achieved an overall accuracy of 0.9556 and a macro‑F1 of 0.9249, confirming reliable binary hazard detection. Unsupervised clustering of the feature space produced three clusters (n\u202f=\u202f3) comprising 27,560 samples. The silhouette score (0.0783) and Davies–Bouldin index (4.9347) suggest weak separation, whereas the Calinski–Harabasz index (103